# SIH26090 — Dynamic Pricing Assistant
### Training pipeline: pretrained perception + a small trained regression model + comparable-listing retrieval

This notebook does 4 things:
1. Loads the artisan pricing reference dataset (`artisan_pricing_dataset.csv`)
2. Trains a **small regression model** to predict price from category/material/complexity/cost features
3. Builds a **sentence-embedding index** (pretrained, no training) so we can retrieve "similar products" at inference time
4. Wraps everything into a single `suggest_price()` function that returns a price band + human-readable reasoning — exactly what the app will call

**Nothing here is trained from scratch except the small regression model in Step 2.** The embedding model and any vision/LLM components are pretrained and used as-is.


## Step 0 — Setup

In [ ]:
# Core installs — scikit-learn, pandas, joblib are pre-installed on Colab.
# sentence-transformers is not, so install it.
!pip install -q sentence-transformers


In [ ]:
import pandas as pd
import numpy as np
import joblib
import json

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## Step 1 — Load the dataset

Upload `artisan_pricing_dataset.csv` when prompted (the file shared in chat — from `/mnt/user-data/outputs/artisan_pricing_dataset.csv`).

If you'd rather not upload manually every run, put the CSV in your Google Drive and mount Drive instead (commented option below).

In [ ]:
from google.colab import files

uploaded = files.upload()  # select artisan_pricing_dataset.csv when prompted
csv_filename = list(uploaded.keys())[0]
df = pd.read_csv(csv_filename)
print(df.shape)
df.head(10)


In [ ]:
# --- Optional alternative: load from Google Drive instead of re-uploading each time ---
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv('/content/drive/MyDrive/SIH26090/artisan_pricing_dataset.csv')


## Step 2 — Explore & sanity-check the data

Quick checks before training: distribution per category, price ranges, any nulls.

In [ ]:
print("Categories:", df['category'].unique())
print()
print(df.groupby('category')['price_inr'].agg(['min','max','mean','count']))
print()
print("Nulls per column:")
print(df.isnull().sum())


In [ ]:
import matplotlib.pyplot as plt

df.boxplot(column='price_inr', by='category', figsize=(10,5), rot=45)
plt.title('Price distribution by category')
plt.suptitle('')
plt.ylabel('Price (INR)')
plt.tight_layout()
plt.show()


## Step 3 — Feature engineering

Features going into the model:
- `category` (one-hot)
- `size_bucket` (one-hot)
- `material` (one-hot)
- `complexity_score` (numeric, 1-5)
- `material_cost_inr` (numeric — this is the artisan-provided or estimated raw material cost)

Target: `price_inr`

Note: `material_cost_inr` is deliberately included as a feature AND used again later as a hard floor in the business-logic layer (Step 6) — the model learns the general relationship, and the floor rule guarantees the final suggestion never dips below a sane multiple of cost even if the model underestimates on a rare combination.

In [ ]:
feature_cols_numeric = ['complexity_score', 'material_cost_inr']
feature_cols_categorical = ['category', 'size_bucket', 'material']
target_col = 'price_inr'

X = df[feature_cols_numeric + feature_cols_categorical]
y = df[target_col]

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), feature_cols_categorical)
    ],
    remainder='passthrough'  # keeps the numeric columns as-is
)


## Step 4 — Train the regression model

Kept deliberately simple: `RandomForestRegressor` with shallow trees, since the dataset is small (~200 rows). A simple, shallow model is also easier to defend to judges than a deep/opaque one — you can literally say "we intentionally kept this simple given data size, and here's how we validated it wasn't overfitting" (see cross-validation below).

In [ ]:
model_pipeline = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=200,
        max_depth=6,          # kept shallow deliberately — small dataset, avoid overfitting
        min_samples_leaf=3,
        random_state=RANDOM_STATE
    ))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

model_pipeline.fit(X_train, y_train)


## Step 5 — Evaluate (and prove it's not overfitting)

In [ ]:
# Hold-out test set performance
y_pred = model_pipeline.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"Test MAE: Rs {mae:.0f}")
print(f"Test R^2: {r2:.3f}")

# K-fold cross-validation on the full dataset — catches overfitting a single train/test split might hide
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(model_pipeline, X, y, cv=kf, scoring='neg_mean_absolute_error')
print(f"\n5-fold CV MAE: Rs {-cv_scores.mean():.0f} (+/- Rs {cv_scores.std():.0f})")


In [ ]:
# Quick visual: predicted vs actual
plt.figure(figsize=(6,6))
plt.scatter(y_test, y_pred, alpha=0.6)
lims = [0, max(y_test.max(), y_pred.max())]
plt.plot(lims, lims, 'r--', label='Perfect prediction')
plt.xlabel('Actual Price (INR)')
plt.ylabel('Predicted Price (INR)')
plt.title('Predicted vs Actual Price')
plt.legend()
plt.show()


## Step 6 — Save the trained model

This `.pkl` file is what your backend (FastAPI/Node service) loads at runtime — no retraining needed in the app itself.

In [ ]:
joblib.dump(model_pipeline, 'pricing_model.pkl')
print("Saved pricing_model.pkl")

# Also save the list of known categories/materials/sizes for input validation in the app
metadata = {
    'categories': sorted(df['category'].unique().tolist()),
    'materials': sorted(df['material'].unique().tolist()),
    'size_buckets': sorted(df['size_bucket'].unique().tolist()),
}
with open('pricing_model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print("Saved pricing_model_metadata.json")
print(metadata)


## Step 7 — Build the "comparable listings" retrieval layer (pretrained, no training)

This uses a pretrained sentence-embedding model (`all-MiniLM-L6-v2`) to embed every product's `description`, so at inference time we can find the top-3 most similar real listings and show them to the artisan as evidence — this is what makes the price suggestion feel trustworthy instead of a black-box number.

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer('all-MiniLM-L6-v2')  # small, fast, pretrained — no training needed

# Embed all reference descriptions once
reference_texts = df['description'].tolist()
reference_embeddings = embedder.encode(reference_texts, show_progress_bar=True, convert_to_numpy=True)

print(reference_embeddings.shape)


In [ ]:
# Save embeddings + the dataframe rows they correspond to, so the app doesn't need to recompute them
np.save('reference_embeddings.npy', reference_embeddings)
df.to_csv('reference_listings.csv', index=False)
print("Saved reference_embeddings.npy and reference_listings.csv")


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def find_comparable_listings(query_description, top_k=3):
    """Returns the top_k most similar reference listings to a new product description."""
    query_embedding = embedder.encode([query_description], convert_to_numpy=True)
    sims = cosine_similarity(query_embedding, reference_embeddings)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]
    results = df.iloc[top_idx][['category', 'description', 'price_inr']].copy()
    results['similarity'] = sims[top_idx]
    return results

# quick test
find_comparable_listings("Handmade cotton block print dupatta, medium size")


## Step 8 — The full pricing function (model + comparables + cost-floor business logic)

This is the function your backend actually calls. It combines:
1. The trained model's raw prediction
2. Retrieved comparable listings (for the "similar products sell for X" reasoning)
3. A **cost-floor rule**: final suggestion never drops below `material_cost x min_markup` for that category — a sanity check the raw model doesn't otherwise guarantee


In [ ]:
# Category-typical minimum markup multipliers, derived from the markup ranges used to build the dataset.
# In production, recompute these from real seller data periodically.
MIN_MARKUP_BY_CATEGORY = {
    'Terracotta_Pottery': 2.0,
    'BlockPrint_Dupatta': 2.5,
    'BlockPrint_Saree': 2.5,
    'Bamboo_Cane_Basket': 2.0,
    'Brass_Metal_Handicraft': 2.0,
    'Wooden_Handicraft': 2.0,
    'Jute_Bag': 2.5,
    'Embroidery_Handloom_Kurta': 2.5,
}

def suggest_price(category, material, size_bucket, complexity_score, material_cost_inr, description):
    """
    Returns a dict with a suggested price band and a human-readable reasoning string.
    This is the function the backend API endpoint should call.
    """
    # 1. Model prediction
    input_df = pd.DataFrame([{
        'complexity_score': complexity_score,
        'material_cost_inr': material_cost_inr,
        'category': category,
        'size_bucket': size_bucket,
        'material': material,
    }])
    model_price = model_pipeline.predict(input_df)[0]

    # 2. Comparable listings
    comparables = find_comparable_listings(description, top_k=3)
    comp_prices = comparables['price_inr'].tolist()
    comp_low, comp_high = (min(comp_prices), max(comp_prices)) if comp_prices else (None, None)

    # 3. Cost floor
    min_markup = MIN_MARKUP_BY_CATEGORY.get(category, 2.0)
    cost_floor = material_cost_inr * min_markup
    final_price = max(model_price, cost_floor)

    # Build a price band (+/- 12% around final estimate, nudged toward comparable range if available)
    low = round(final_price * 0.88, -1)
    high = round(final_price * 1.12, -1)
    if comp_low is not None:
        low = round((low + comp_low) / 2, -1)
        high = round((high + comp_high) / 2, -1)

    reasoning = (
        f"Suggested price: Rs {int(low)}-{int(high)}. "
        f"Based on your material cost of Rs {int(material_cost_inr)} "
        f"(typical markup for {category.replace('_',' ')}: {min_markup}x), "
        f"and {len(comp_prices)} similar products currently priced around "
        f"Rs {int(comp_low) if comp_low else '-'}-{int(comp_high) if comp_high else '-'}."
    )

    return {
        'suggested_low': int(low),
        'suggested_high': int(high),
        'model_raw_prediction': round(float(model_price), 2),
        'cost_floor': round(cost_floor, 2),
        'comparable_listings': comparables.to_dict(orient='records'),
        'reasoning': reasoning,
    }


## Step 9 — Test the full pipeline end-to-end

In [ ]:
result = suggest_price(
    category='BlockPrint_Dupatta',
    material='Cotton',
    size_bucket='Medium',
    complexity_score=3,
    material_cost_inr=150,
    description='Handmade cotton block print dupatta with floral pattern, medium size'
)

print(result['reasoning'])
print()
for k, v in result.items():
    if k != 'comparable_listings':
        print(f"{k}: {v}")
print("\nComparable listings used:")
for c in result['comparable_listings']:
    print(c)


In [ ]:
# Try another category to sanity-check the cost floor kicks in correctly
result2 = suggest_price(
    category='Wooden_Handicraft',
    material='Sheesham Wood',
    size_bucket='Large',
    complexity_score=5,
    material_cost_inr=1200,
    description='Hand carved wooden elephant showpiece, large size, intricate detailing'
)
print(result2['reasoning'])


## Step 10 — Download the artifacts for your backend

Run this to download everything your FastAPI/Node backend needs:
- `pricing_model.pkl` — trained regression model
- `pricing_model_metadata.json` — valid categories/materials/sizes
- `reference_embeddings.npy` + `reference_listings.csv` — for the comparable-listings lookup

In your backend, load these once at startup (not per-request) and reuse the `embedder = SentenceTransformer('all-MiniLM-L6-v2')` object + `suggest_price()` logic shown above.

In [ ]:
from google.colab import files

files.download('pricing_model.pkl')
files.download('pricing_model_metadata.json')
files.download('reference_embeddings.npy')
files.download('reference_listings.csv')


---
### Next steps for the team
1. Swap in real scraped/manually-verified listings for at least a few rows per category before the final demo — strengthens your "we validated against real data" claim.
2. Wire `suggest_price()` into a FastAPI endpoint (`POST /suggest-price`) that the mobile app calls after the Image Studio + Auto-Cataloger steps produce a description.
3. In the app UI, surface the `reasoning` string and the comparable listings directly — that transparency is your strongest trust-building feature for judges.
4. If you have time left, retrain periodically (e.g., nightly) as more real transactions come in — mention this as your "Phase 2" roadmap in the pitch.
